# **Preprocesamiento para PySpark**

En esta sección se realiza el preprocesamiento de los datos para el flujo de modelado con PySpark.

Se trabaja con el dataset completo en formato distribuido, evitando cualquier conversión a pandas o transferencia masiva de datos al driver.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import when, col

In [2]:
spark = SparkSession.builder.master("local[*]").appName("PreprocesamientoPySpark").getOrCreate()

## **Carga del dataset completo en Spark**

El dataset se carga directamente en una `SparkSession`.

In [3]:
data_path = r"C:\Users\Daniel Rangel\Documents\MachineLearning\Data\accepted_2007_to_2018Q4.csv.gz"

df_spark = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(data_path)
)

df_spark = df_spark.withColumn(
    "default",
    when(col("loan_status") == "Charged Off", 1).otherwise(0)
)

print("Número de columnas:", len(df_spark.columns))
print("Número de filas:", df_spark.count())

Número de columnas: 152
Número de filas: 2260701


## **Selección de variables**

Se seleccionan las mismas variables utilizadas en el flujo de scikit-learn, con el fin de mantener consistencia entre ambos enfoques y permitir una comparación más justa del desempeño de las herramientas.

In [4]:
num_vars = [
    "loan_amnt",
    "int_rate",
    "fico_range_high",
    "annual_inc",
    "dti",
    "revol_util",
    "open_acc",
    "total_acc"
]

cat_vars = [
    "emp_length",
    "purpose",
    "home_ownership",
    "addr_state"
]

selected_vars = num_vars + cat_vars + ["default"]

df_spark_model = df_spark.select(*selected_vars)

print("Columnas seleccionadas:")
print(selected_vars)
print("Número de columnas seleccionadas:", len(df_spark_model.columns))
print("Número de filas:", df_spark_model.count())

Columnas seleccionadas:
['loan_amnt', 'int_rate', 'fico_range_high', 'annual_inc', 'dti', 'revol_util', 'open_acc', 'total_acc', 'emp_length', 'purpose', 'home_ownership', 'addr_state', 'default']
Número de columnas seleccionadas: 13
Número de filas: 2260701


## **Codificación de variables categóricas en PySpark**

Siguiendo la guía del proyecto, las variables categóricas se codifican utilizando `StringIndexer` y `OneHotEncoder`.

Este proceso permite transformar las categorías textuales en una representación numérica adecuada para el modelado, manteniendo siempre los datos en formato DataFrame de Spark.

In [5]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [6]:
indexers = [
    StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep"
    )
    for col_name in cat_vars
]

encoder = OneHotEncoder(
    inputCols=[f"{col_name}_indexed" for col_name in cat_vars],
    outputCols=[f"{col_name}_encoded" for col_name in cat_vars],
    handleInvalid="keep"
)

In [7]:
print("Variables categóricas originales:", cat_vars)
print("Columnas indexadas:", [f"{col_name}_indexed" for col_name in cat_vars])
print("Columnas codificadas:", [f"{col_name}_encoded" for col_name in cat_vars])

Variables categóricas originales: ['emp_length', 'purpose', 'home_ownership', 'addr_state']
Columnas indexadas: ['emp_length_indexed', 'purpose_indexed', 'home_ownership_indexed', 'addr_state_indexed']
Columnas codificadas: ['emp_length_encoded', 'purpose_encoded', 'home_ownership_encoded', 'addr_state_encoded']


### **Corrección de tipos de datos**

Antes de aplicar el pipeline de transformación, es necesario asegurar que las variables numéricas se encuentren en formato numérico dentro de Spark, ya que algunas columnas fueron interpretadas como tipo `string` durante la lectura del archivo.

In [8]:
from pyspark.sql.functions import col

for c in num_vars:
    df_spark_model = df_spark_model.withColumn(c, col(c).cast("double"))

df_spark_model.printSchema()

root
 |-- loan_amnt: double (nullable = true)
 |-- int_rate: double (nullable = true)
 |-- fico_range_high: double (nullable = true)
 |-- annual_inc: double (nullable = true)
 |-- dti: double (nullable = true)
 |-- revol_util: double (nullable = true)
 |-- open_acc: double (nullable = true)
 |-- total_acc: double (nullable = true)
 |-- emp_length: string (nullable = true)
 |-- purpose: string (nullable = true)
 |-- home_ownership: string (nullable = true)
 |-- addr_state: string (nullable = true)
 |-- default: integer (nullable = false)



## **Tratamiento de valores faltantes en PySpark**

Antes de ensamblar las variables en un vector de características, es necesario imputar los valores faltantes de las variables numéricas.

En este caso, se utilizará imputación por mediana o media según la disponibilidad de las herramientas de PySpark, con el fin de evitar valores `NaN` en el vector final de características.

In [9]:
from pyspark.ml.feature import Imputer

In [10]:
imputer = Imputer(
    inputCols=num_vars,
    outputCols=[f"{c}_imputed" for c in num_vars]
).setStrategy("median")

In [11]:
print("Columnas numéricas originales:", num_vars)
print("Columnas numéricas imputadas:", [f"{c}_imputed" for c in num_vars])

Columnas numéricas originales: ['loan_amnt', 'int_rate', 'fico_range_high', 'annual_inc', 'dti', 'revol_util', 'open_acc', 'total_acc']
Columnas numéricas imputadas: ['loan_amnt_imputed', 'int_rate_imputed', 'fico_range_high_imputed', 'annual_inc_imputed', 'dti_imputed', 'revol_util_imputed', 'open_acc_imputed', 'total_acc_imputed']


## **Ensamblado de variables**

Una vez codificadas las variables categóricas, se ensamblan las variables numéricas y categóricas transformadas en un único vector de características mediante `VectorAssembler`.

Posteriormente, se aplica `StandardScaler` con el fin de obtener una representación escalada de las variables numéricas y codificadas. Finalmente, el DataFrame resultante se almacenará en memoria para optimizar el entrenamiento posterior del modelo.

In [12]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml import Pipeline
from pyspark import StorageLevel

assembler_inputs = [f"{c}_imputed" for c in num_vars] + [f"{col_name}_encoded" for col_name in cat_vars]

assembler = VectorAssembler(
    inputCols=assembler_inputs,
    outputCol="features",
    handleInvalid="keep"
)

pipeline = Pipeline(stages=indexers + [encoder, imputer, assembler])

In [13]:
pipeline_model = pipeline.fit(df_spark_model)
df_spark_prepared = pipeline_model.transform(df_spark_model)

## **Almacenamiento en memoria del DataFrame transformado**

Siguiendo la guía del proyecto, el DataFrame resultante se almacena en memoria después del ensamblado de variables y antes del entrenamiento del modelo, con el fin de optimizar el procesamiento posterior en PySpark.

In [14]:
from pyspark import StorageLevel

df_spark_prepared = df_spark_prepared.cache()

In [15]:
df_spark_prepared.select("default").show(5)

+-------+
|default|
+-------+
|      0|
|      0|
|      0|
|      0|
|      0|
+-------+
only showing top 5 rows



## **División en entrenamiento y prueba**

Una vez preparado y almacenado en memoria el DataFrame transformado, se realiza la partición en conjuntos de entrenamiento y prueba con una proporción 80/20, manteniendo siempre los datos en formato DataFrame de Spark.

In [16]:
train_spark, test_spark = df_spark_prepared.randomSplit([0.8, 0.2], seed=42)

print("Filas train:", train_spark.count())
print("Filas test:", test_spark.count())

Filas train: 1808391
Filas test: 452310


## **Resultado de la partición en PySpark**

La partición del dataset en PySpark produjo **1,808,391** observaciones para entrenamiento y **452,310** para prueba, lo cual corresponde aproximadamente a una división 80/20, en línea con la guía del proyecto.

A diferencia del flujo en scikit-learn, la partición se realizó mediante `randomSplit`, ya que la estratificación exacta no forma parte del procedimiento estándar de partición en PySpark. Por esta razón, resulta conveniente verificar posteriormente la distribución de la variable objetivo en ambos subconjuntos.

In [17]:
train_dist = train_spark.groupBy("default").count().orderBy("default")
test_dist = test_spark.groupBy("default").count().orderBy("default")

print("Distribución en train:")
train_dist.show()

print("Distribución en test:")
test_dist.show()

Distribución en train:
+-------+-------+
|default|  count|
+-------+-------+
|      0|1593823|
|      1| 214568|
+-------+-------+

Distribución en test:
+-------+------+
|default| count|
+-------+------+
|      0|398320|
|      1| 53990|
+-------+------+



### **Verificación de la distribución de clases**

La distribución de la variable objetivo en los conjuntos de entrenamiento y prueba se mantiene muy similar a la observada en el dataset original.

En el conjunto de entrenamiento, la clase `0` continúa siendo ampliamente mayoritaria frente a la clase `1`, y el mismo patrón se observa en el conjunto de prueba. Esto indica que, aunque `randomSplit` no garantiza una estratificación exacta, en este caso la partición conservó de manera adecuada el desbalance de clases del problema.

Por lo tanto, los subconjuntos generados en PySpark resultan apropiados para continuar con la etapa de modelado y comparación con el flujo implementado en scikit-learn.

## **Conclusión de la sección**

En esta sección se realizó el preprocesamiento del dataset completo en PySpark, manteniendo los datos en formato distribuido en todo momento. Se seleccionaron variables relevantes, se corrigieron tipos de datos, se imputaron valores faltantes en variables numéricas, se codificaron variables categóricas mediante `StringIndexer` y `OneHotEncoder`, y se ensamblaron las características en un vector final para modelado.

Asimismo, se almacenó el DataFrame transformado en memoria y se realizó la partición en entrenamiento y prueba, conservando de manera adecuada la distribución de la variable objetivo. Con ello, el flujo de PySpark queda preparado para la etapa de entrenamiento del modelo.